In [ ]:
import json
import math
import pandas as pd

def get_answer_confidence(logprobs_content, correct_answer):
    """Get probability of correct answer token from first meaningful token."""
    # Skip BOS token (empty token at position 0)
    first_real = logprobs_content[1] if logprobs_content[0]['token'] == '' else logprobs_content[0]
    
    # Check top_logprobs for correct answer token
    for tl in first_real['top_logprobs']:
        if tl['token'].strip() == str(correct_answer).strip():
            return math.exp(tl['logprob'])
    
    # If correct answer not in top_logprobs, return 0
    return 0.0

def analyze_confidences(data_list):
    rows = []
    for data in data_list:
        row = {'image_id': data['image_id'], 'question_id': data['question_id']}
        for q_type, logprobs in data['generated_logits'].items():
            correct = data['answers'][q_type]
            conf = get_answer_confidence(logprobs['content'], correct)
            row[f'conf_{q_type}'] = conf
            row[f'pred_{q_type}'] = logprobs['content'][1]['token'].strip()  # predicted token
            row[f'correct_{q_type}'] = str(correct)
            row[f'acc_{q_type}'] = row[f'pred_{q_type}'] == str(correct)
        rows.append(row)
    return pd.DataFrame(rows)

# Usage
with open('/home/david/Desktop/yuna/HPA/evaluation/logits/pretrained/llava-v1.6-vicuna-7b-hf/vqa_1k_control.jsonl') as f:
    data_list = [json.loads(l) for l in f]

df = analyze_confidences(data_list)

# Summary per question type
q_types = ['question', 'deictic_removed', 'object_removed', 'weaker_object', 'subject_ablated']
for qt in q_types:
    print(f"{qt}: avg_conf={df[f'conf_{qt}'].mean():.3f}, acc={df[f'acc_{qt}'].mean():.3f}")